# `ptof_obs_nightly_baseline`

## What this notebook does
Recomputes the two statistical baselines that detection depends on to tell "normal" from
"anomalous": per-capability latency percentiles, and per-capability expected response schema.
Nothing in this notebook detects anything itself -- it produces the reference points other
notebooks compare live data against.

## Position in the pipeline
- **Separate scheduled job** (`obs_nightly_baseline`, job id `428356310089497`) -- runs nightly,
  independently of `obs_fresh_scan`. Not one of `obs_fresh_scan`'s 6 tasks.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`) and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`, joined here as `r.active = true`
  so an inactive/decommissioned capability doesn't pollute a baseline).
- **Downstream:** `ptof_obs_latency_detection` reads `capability_latency_baseline` to decide
  whether a call is a latency anomaly relative to that capability's own history (not a fixed SLA).
  `ptof_obs_mal_output` reads `response_schema_baseline` to detect schema drift (a field that used
  to reliably appear has gone missing).

## Why baselines are computed nightly, separately from detection
Both queries do a `CREATE OR REPLACE` over a 30-day rolling window -- expensive relative to the
hourly detection queries, and baselines shouldn't jitter run-to-run the way live detection does.
Splitting this into its own nightly job means detection runs (every `obs_fresh_scan` tick) stay
cheap and compare against a stable reference point instead of recomputing it every time.

## Tables/views touched
- **Reads:** `mq_gmdf_dev.oil_obs.v_llm_bronze`, `mq_gmdf_dev.oil_obs.capability_registry`.
- **Writes:** `mq_gmdf_dev.oil_obs.capability_latency_baseline` (per-capability p50/p95/p99 latency
  and a computed anomaly upper bound), `mq_gmdf_dev.oil_obs.response_schema_baseline` (per-capability
  expected JSON schema DDL, inferred from a sample of successful, non-blank recent responses).


In [0]:
%sql
-- capability_latency_baseline: gives ptof_obs_latency_detection a per-capability "normal" to
-- compare against, instead of one fixed SLA for every capability -- a capability that's
-- naturally slower (bigger prompts, heavier model) shouldn't trip the same threshold as a fast
-- one.
-- Two guards, both learned the hard way:
--   n_samples >= 30      — dsa_batch_summary had n=1, so p50=p95=p99=bound and 2826ms fired
--                          as an "anomaly" when dsa_copilot's p95 is 3963ms
--   baseline_span_days >= 7 — the Jun 12 → Aug 12 blackout means a nominal 30-day window holds
--                          only ~1.5 days of data, so n alone is not evidence of stability
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_latency_baseline AS
SELECT
    b.capability,
    -- p50/p95/p99: the shape of "normal" latency for this capability specifically.
    approx_percentile(b.latency_ms, 0.50) AS p50_ms,
    approx_percentile(b.latency_ms, 0.95) AS p95_ms,
    approx_percentile(b.latency_ms, 0.99) AS p99_ms,
    -- anomaly_upper_bound_ms: p95 + 3*IQR -- a standard outlier-fence formula, computed once here
    -- so ptof_obs_latency_detection doesn't need to recompute the percentile spread on every run.
    approx_percentile(b.latency_ms, 0.95)
      + 3 * (approx_percentile(b.latency_ms, 0.75)
             - approx_percentile(b.latency_ms, 0.25))   AS anomaly_upper_bound_ms,
    count(*)                                            AS n_samples,
    datediff(max(b.called_at), min(b.called_at))        AS baseline_span_days,
    -- is_reliable: the flag ptof_obs_latency_detection actually gates on -- latency_anomaly can't
    -- fire for a capability until this is true, which is why some capabilities show 0 anomalies
    -- while genuinely data-starved rather than genuinely healthy.
    (count(*) >= 30
     AND datediff(max(b.called_at), min(b.called_at)) >= 7) AS is_reliable,
    min(b.called_at)                                    AS baseline_from,
    max(b.called_at)                                    AS baseline_through,
    current_timestamp()                                 AS computed_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  -- r.active = true: an inactive/decommissioned capability's historical calls shouldn't set a
  -- baseline anyone will ever compare live traffic against.
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  -- exclude known-fast-failing and mock-transport calls -- they'd drag the percentiles toward
  -- near-zero and make a genuinely slow real call look less anomalous than it is.
  AND b.is_credential_fastfail = false
  AND b.transport             <> 'mock'
  AND b.success                = true
GROUP BY b.capability;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- response_schema_baseline: gives ptof_obs_mal_output a per-capability "what fields normally
-- appear" reference, so it can detect a field silently disappearing from responses (schema
-- drift) rather than only checking for blank/malformed output.
-- The old upper bound (current_timestamp() - INTERVAL 1 DAYS) excluded every dsa_* capability,
-- since that data begins 2026-08-18 01:30. That is why the baseline covered 3 of 5.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.response_schema_baseline (
    capability       STRING,
    schema_ddl       STRING,
    sample_count     BIGINT,
    baseline_from    TIMESTAMP,
    baseline_through TIMESTAMP
);

INSERT OVERWRITE TABLE mq_gmdf_dev.oil_obs.response_schema_baseline
SELECT
    b.capability,
    -- schema_ddl: the inferred JSON schema across a sample of recent responses -- the reference
    -- shape ptof_obs_mal_output's response_schema_drift compares each capability's current
    -- responses against, field by field.
    schema_of_json_agg(cast(b.response_parsed AS STRING)) AS schema_ddl,
    count(*)         AS sample_count,
    min(b.called_at) AS baseline_from,
    max(b.called_at) AS baseline_through
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.success = true
  -- exclude blank/credential-fastfail responses from the sample -- a schema inferred from empty
  -- or error responses would be garbage, not a real "expected shape."
  AND b.is_blank_output        = false
  AND b.is_credential_fastfail = false
  AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
GROUP BY b.capability
-- HAVING count(*) >= 20: same spirit as latency's n_samples >= 30 guard -- don't infer a
-- "reliable" schema baseline from a handful of calls.
HAVING count(*) >= 20;

num_affected_rows,num_inserted_rows
1,1
